# AgentDrift · Part B — Preparing the data

*EDA 2 / 3 · Effect of the preprocessing · ESILV 2026, group DIA 7*

---

### Where we are

Our guardrail watches an AI agent step by step and must stop it **before it executes an action dictated by a hidden instruction** (an *indirect prompt injection*). Part A (`01_eda.ipynb`) showed that the corpus contains **shortcuts**: a model could detect attacks without reading what the agent does, mainly by recognising the simulated world a trajectory takes place in, or the wording tics of the generator.

This notebook checks the **preparation** decided in part A. A good preparation must do two things at once:
- **remove the shortcuts**, so that the score of a model reflects what it would do on a real agent;
- **keep the real signal**: the injected instruction, and the relations a detector needs (who receives the data, where an argument comes from).

### The three preparations

| Preparation | What it does | Status |
|---|---|---|
| **Anonymisation** | replaces names, companies, addresses and domains by placeholders that keep relations but not identities | done — §1 to §4 |
| **Masking of generator tics** | hides typical openers ("CORRECTION:", "Per policy…"), policy codes and amounts | in development — §5 |
| **Hand annotations** | six risk categories for the 103 tools; is the injection readable by a human? (300 cases) | draft ready, under review — §6, §7 |

### Contents

1. What the anonymisation changes · 2. Are there identifiers left? · 3. Can the text still reveal the world? · 4. Is the useful signal kept? · 5. Masking the generator's tics · 6. Tool risk categories · 7. Is the injection readable by a human? · **Summary** · Reproducibility check

> **Ground rules.** The probes are fitted on train and measured on validation; the test set is never used. The crossing of tool risk with the attack labels stays hidden until the risk table is validated, so that the reviewer cannot adjust the categories to the answers. Every number quoted in the text is recomputed by the last cell.

In [1]:
import warnings

import pandas as pd

from agentdrift.data import annotations as N
from agentdrift.data import audit as A
from agentdrift.data.anonymize import placeholder_counts
from agentdrift.pipelines.run_eda import compute_b
from agentdrift.utils.seed import set_seed
from agentdrift.viz import notebook as NB
from agentdrift.viz import plots as P

warnings.filterwarnings("ignore", category=FutureWarning)
set_seed()
P.setup()

d = A.load_inputs()          # raw tables
da = A.load_inputs_anon()    # anonymised tables: the only input of every model
rb = compute_b(d, da)        # all part B results (shared with `run_eda --part B`)

## 1. What does the anonymisation change in the text?

**Why it matters.** In part A, the simulated world (the fictitious user, company and contacts) revealed whether a trajectory was attacked. We must remove that identity **without erasing the attack itself**: the guardrail still has to see the injected instruction and to know whether the data goes to the user, to a known contact, or to an unknown outside address.

Below, the same injection step before and after anonymisation, for each attack type (examples drawn at random from the training part).

In [2]:
ex = rb["text_before_after"]
ex.style.set_properties(**{"text-align": "left", "white-space": "pre-wrap", "max-width": "520px"})

In [3]:
ph = placeholder_counts(da["steps"])
NB.table(ph.rename("occurrences").to_frame(), int_cols=["occurrences"], caption="Placeholders in the anonymised text")

,occurrences
,"18,765"
,"18,327"
,"10,744"
,"9,610"
,"4,774"
,"3,014"
,"2,507"
,"1,285"
,"1,078"
,223


**What we see**
- Identities become **placeholders that keep the relation**: `<USER>` and `<USER_EMAIL>` for the user, `<CONTACT_k>` for the k-th contact of the world, `<INT_EMAIL_k>` for an unknown address *inside* the user's company and `<EXT_EMAIL_k>` for one *outside*. The most frequent is `<CONTACT_k>` (18,765 occurrences).
- **The attack content is untouched**: the opener, the instruction, the amounts and account numbers stay readable. Only *who* is targeted is abstracted.
- Numbers are given in reading order (task, then each step), so the text of a step never depends on what comes after it.

**What we do.** From now on, every model reads the **anonymised** tables only. The masking of §5 will be applied on top of them.

<sub>Traceability: leak L9 (world identity) · `data/anonymize.py`</sub>

## 2. Are there identifiers left?

**Why it matters.** A single forgotten name or address is enough for a model to recognise the world again. We search every trajectory for the names, companies, addresses and domain labels of its own world, before and after anonymisation.

In [4]:
res = rb["residual"]
NB.show(P.plot_before_after_bars(res[["raw", "anonymised"]], "World identifiers left in the text",
                                 "share of trajectories containing the identifier"), "eda_b1_residual_identifiers")
NB.table(res, pct_cols=["raw", "anonymised"], pct_digits=2, int_cols=["n residual (anonymised)"])

<Figure size 825x385 with 1 Axes>

,raw,anonymised,n residual (anonymised)
full name,40.91%,0.00%,0
world e-mail,15.09%,0.00%,0
company name,26.64%,0.00%,0
domain label,43.66%,0.02%,3


**What we see**
- Before anonymisation, 41% of trajectories contain a full name from their world, 44% its domain label, 27% its company name and 15% one of its addresses.
- After: **no trajectory** keeps a name, address or company. 3 of 12,536 still contain the domain label, inside longer words (a health insurer's name, a password, a class name): false alarms of the search, checked by hand.

**What we do.** The first criterion is met. An automatic test keeps it true whenever the anonymisation code changes.

<sub>Traceability: `tests/test_anonymize.py::test_corpus_has_no_world_identity_left`</sub>

## 3. Can the text still reveal the world?

**Why it matters.** Removing names is not enough if the rest of the text still points to one world (a recurring product, a typical phrasing). This is the real success criterion of the anonymisation.

**The probe.** For each validation trajectory, we look for the most similar training trajectory (word-based similarity) and check whether it comes from the same world. If the text reveals the world, the nearest neighbour often shares it.

**The fair chance level.** Worlds were generated together with an attack type, so two trajectories of the same type are naturally close. The fair comparison is therefore the chance of hitting the right world **among worlds of the same domain and attack type**.

In [5]:
wp = rb["world_probe"]
NB.show(P.plot_world_probe(wp), "eda_b2_world_probe")
NB.table(wp.rename(columns={"n_eval": "validation trajectories", "world_accuracy": "same world as nearest neighbour",
                            "chance_given_domain": "chance (same domain)", "chance_given_category": "chance (same domain and type)",
                            "neighbour_same_category": "neighbour of the same type"}),
         pct_cols=["same world as nearest neighbour", "chance (same domain)", "chance (same domain and type)",
                   "neighbour of the same type"], int_cols=["validation trajectories"])

<Figure size 825x286 with 1 Axes>

,validation trajectories,same world as nearest neighbour,chance (same domain),chance (same domain and type),neighbour of the same type
raw,"1,723",12.2%,0.7%,2.1%,76.8%
anonymised,"1,723",3.1%,0.7%,2.1%,79.1%


**What we see**
- On the raw text, the nearest neighbour comes from the same world for **12.2%** of the 1,723 validation trajectories: 5.9 times the fair chance level (2.1%).
- After anonymisation: **3.1%**, i.e. 1.5 times chance. The probe has lost 75% of its accuracy.
- The small residue most likely comes from **content, not identity**: the neighbour has the same attack type in 79% of cases, and since worlds were generated per attack type, matching the content of an attack also narrows down the world.

**What we do.** The world identity is no longer readable from the text, so the split by task template (which cannot also separate worlds) is acceptable. The probe will be re-run after the masking of §5, which should lower the residue further.

<sub>Traceability: leak L9 · `docs/CONTEXT.md` §8.1 · `ai_governance/AI_AUDIT.md` A1 (why the world lookup of part A is not the criterion)</sub>

## 4. Is the useful signal kept?

**Why it matters.** Part A showed that the gate's best clue, especially for partial attacks, is **where the arguments of an action come from**: an outside recipient, a value copied from an earlier tool response. These indicators rely on addresses, which the anonymisation has just replaced. We recompute them on the placeholders and compare.

In [6]:
prov = rb["provenance"]
NB.show(P.plot_provenance_before_after(prov), "eda_b3_provenance")
delta = prov.xs("anonymised", axis=1, level=1) - prov.xs("raw", axis=1, level=1)
delta.style.format("{:+.2%}").set_caption("Change after anonymisation (share of steps)")

<Figure size 1320x453.2 with 2 Axes>

,amount in args,arg from previous obs,external domain,unknown address,unknown same-domain address
label,,,,,
B,-0.10%,-0.13%,+0.00%,+0.00%,+0.00%
I,-0.23%,-0.13%,+0.00%,+0.00%,+0.00%
F,-0.33%,-0.20%,+0.00%,+0.00%,+0.00%
H full,-0.10%,-0.22%,-0.30%,-0.30%,+0.00%
H partial,-0.06%,-0.17%,+0.00%,+0.00%,+0.00%
H delayed,-0.11%,+0.00%,-0.11%,-0.11%,+0.00%


Groups: **B** normal steps, **I** injection points, **F** failed injections, **H** hijacked steps by attack type.

**What we see**
- No indicator moves by more than **0.33 points** (the largest change: amounts in the arguments of failed injections).
- The contrasts that matter are intact: hijacked actions of full attacks send to an outside domain in 43% of cases against 2% of normal steps; arguments of partial attacks come from an earlier response in 58% of cases against 19%.
- The "new address inside the company" relation, carried by `<INT_EMAIL_k>`, is unchanged.

**What we do.** Provenance features are computed on the anonymised text with no loss. **The anonymisation removes identities, not relations.**

<sub>Traceability: hypotheses H1, H8 (provenance and task–action coherence)</sub>

## 5. Masking the generator's tics

**Why it matters.** In part A, each attack type had its own opener ("CORRECTION:" for partial attacks, "Per policy…" for full ones, "SECURITY ALERT" for failed ones). A model that learns these tics detects the *generator*, not the *attack*, and would fail on real injections.

**The rule.** Rather than masking the list of openers found in part A (that list was drawn from the whole corpus, test included, and would miss new variants), the masking applies a **general rule**: any upper-case label followed by ":", any "Per policy *code*", and amounts become placeholders. A second variant also removes the agent's reasoning.

**Status.** The masking module is being developed in the features package (Jira KAN-19). When it is delivered, this section will show the masked text next to the anonymised one, and the cheating baselines of part A recomputed on it.

In [7]:
status = pd.Series({
    "masking module available": rb["masking_available"],
    "tool risk table": rb["tool_risk_status"]["status"],
    "injections annotated": f"{int(rb['annotation_progress']['annotated'].sum())} / {int(rb['annotation_progress']['sampled'].sum())}",
}, name="status").to_frame()
status

,status
masking module available,False
tool risk table,draft
injections annotated,0 / 300


## 6. Tool risk categories

**Why it matters.** Each domain has its own tools, so a model that learns tool names cannot be used on a new application (part A, §6). Six **risk categories** describe what a tool does rather than what it is called, and can be applied to any agent:

| Category | Meaning |
|---|---|
| read | reads data without changing anything |
| internal write | modifies something inside the user's own space |
| external send | sends data outside |
| payment | moves money or commits the user financially |
| delete | deletes data |
| execute | runs code or commands |

Each tool was categorised from its name and description only, never from the attack labels. The draft (prepared with AI assistance) flags the borderline cases for the human review.

In [8]:
tr = N.read_tool_risk()
counts = rb["tool_risk_counts"]
NB.table(counts.rename("tools").to_frame().T, int_cols=counts.index, caption="Tools per risk category (draft)")

risk,read,internal_write,external_send,payment,delete,execute
tools,51,29,11,5,4,3


In [9]:
NB.table(tr[tr["ambiguous"]][["tool", "domain", "risk"]], index=False, caption="Borderline tools submitted to the review")

tool,domain,risk
add_beneficiary,banking,payment
apply_loan,banking,payment
convert_currency,banking,read
dispute_transaction,banking,external_send
open_savings,banking,internal_write
create_pr,coding,internal_write
debug_error,coding,read
git_pull,coding,internal_write
git_push,coding,external_send
create_filter,email,internal_write


In [10]:
if rb["tool_risk_by_group"] is not None:   # shown only once the table is validated
    NB.show(P.plot_table_heatmap(rb["tool_risk_by_group"].drop(columns="n"),
                                 "Risk category of the proposed action, by step group"), "eda_b4_tool_risk")

**Where we are.** 103 tools in six categories: 51 read, 29 internal write, 11 external send, 5 payment, 4 delete, 3 execute. **20 tools are borderline** (for example `add_beneficiary`: no payment yet, but it enables a future transfer) and await the review (Jira KAN-58).

**What comes next.** Once the table is validated, the cell above shows which risk categories hijacked actions use compared with normal ones. This crossing is deliberately hidden until then: seeing it would tempt the reviewer to move a tool to the category that "works best".

<sub>Traceability: hypotheses H1, H5 (transfer to an unseen domain) · `configs/annotations/tool_risk.yaml`</sub>

## 7. Is the injection readable by a human?

**Why it matters.** Part A estimated with simple markers that a large share of partial and delayed injections cannot be read in the tool response — which would make the scanner blind to them and leave the gate alone. Simple markers can be wrong in both directions, so a human reads a sample.

**The protocol.** 300 injection steps (100 per attack type, from train and validation, drawn with a fixed seed) are annotated **blind**: shuffled, without the attack type and without the trajectory identifier, so that the annotator cannot know whether the attack succeeded. The answer is yes or no: *would an instruction addressed to the agent be recognisable here?*

In [11]:
NB.table(rb["annotation_progress"], pct_cols=["share_oui"], caption="Annotation progress per attack type")

,sampled,annotated,share_oui
source_category,,,
attacked_full,100,0,
attacked_partial,100,0,
attacked_delayed,100,0,


In [12]:
vvh = rb["visible_vs_heuristic"]
if vvh is not None:   # shown once rows are annotated
    NB.show(P.plot_table_heatmap(vvh[["visible (human)", "visible (regex union)", "agreement"]],
                                 "Is the injection readable? Human reader against simple markers"), "eda_b5_injection_visible")
    NB.table(vvh, pct_cols=["visible (human)", "visible (regex union)", "agreement"], dec_cols={"kappa": 2})

**Where we are.** The annotation sheet is ready; no row is annotated yet (0 of 300, Jira KAN-58).

**What comes next.** The cell above will compare the human reading with the markers of part A, attack type by attack type, with an agreement score corrected for chance (Cohen's kappa). The result decides how the scanner's recall is reported: on all injections, or separately on readable and unreadable ones.

<sub>Traceability: hypotheses H2 (scanner and gate are complementary), H8 · `configs/annotations/README.md`</sub>

## Summary

| Check | Result | Status |
|---|---|---|
| Identifiers left after anonymisation | none (3 false alarms inside longer words) | ✅ done |
| Can the text reveal the world? | 12.2% → 3.1%, close to the fair chance level of 2.1% | ✅ done |
| Is the useful signal kept? | every indicator within 0.33 points | ✅ done |
| Masking of generator tics | general rule defined | ⏳ in development (KAN-19) |
| Tool risk categories | draft: 6 categories, 20 borderline tools | ⏳ under review (KAN-58) |
| Is the injection readable? | blind sheet of 300 cases | ⏳ to annotate (KAN-58) |

**Decision.** The anonymised tables are the only input of every model. The masking and the annotations complete the preparation as soon as they are delivered; this notebook will then be re-run and §5 to §7 completed.

**Next:** part C (`03_eda_protocol.ipynb`) sets how the models will be evaluated.

## Reproducibility check

This cell recomputes every number quoted in the text and stops the notebook if one of them no longer matches the data.

In [13]:
from agentdrift.viz.notebook import num, pct

worst = delta.abs().stack().idxmax()
raw_acc, anon_acc = wp.loc["raw", "world_accuracy"], wp.loc["anonymised", "world_accuracy"]
chance = wp.loc["raw", "chance_given_category"]
po = lambda g, c: pct(prov.loc[g, (c, "anonymised")])
NB.check([
    ("§1 most frequent placeholder", f"{ph.index[0]} ({num(ph.iloc[0])})", "<CONTACT_k> (18,765)"),
    ("§2 before: name / domain label / company / address",
     " / ".join(pct(res.loc[k, "raw"]) for k in ("full name", "domain label", "company name", "world e-mail")), "41% / 44% / 27% / 15%"),
    ("§2 after: name + address + company", num(res["n residual (anonymised)"].drop("domain label").sum()), "0"),
    ("§2 after: domain label", f"{res.loc['domain label', 'n residual (anonymised)']:.0f} of {num(len(d['trajectories']))}", "3 of 12,536"),
    ("§3 probe raw / anonymised", f"{pct(raw_acc, 1)} / {pct(anon_acc, 1)}", "12.2% / 3.1%"),
    ("§3 fair chance level", pct(chance, 1), "2.1%"),
    ("§3 validation trajectories", num(wp.loc["raw", "n_eval"]), "1,723"),
    ("§3 times chance raw / anonymised", f"{raw_acc / chance:.1f} / {anon_acc / chance:.1f}", "5.9 / 1.5"),
    ("§3 accuracy lost", pct(1 - anon_acc / raw_acc), "75%"),
    ("§3 neighbour of the same type", pct(wp.loc["anonymised", "neighbour_same_category"]), "79%"),
    ("§4 largest change (points)", f"{100 * delta.abs().to_numpy().max():.2f}", "0.33"),
    ("§4 largest change: where", f"{worst[1]} / {worst[0]}", "amount in args / F"),
    ("§4 outside domain H full / B", f"{po('H full', 'external domain')} / {po('B', 'external domain')}", "43% / 2%"),
    ("§4 earlier response H partial / B", f"{po('H partial', 'arg from previous obs')} / {po('B', 'arg from previous obs')}", "58% / 19%"),
    ("§4 inside-company relation unchanged", f"{100 * delta['unknown same-domain address'].abs().max():.2f}", "0.00"),
    ("§6 tools per category", " ".join(f"{v}" for v in counts), "51 29 11 5 4 3"),
    ("§6 borderline tools", str(rb["tool_risk_status"]["n_ambiguous"]), "20"),
    ("§6 status", rb["tool_risk_status"]["status"], "draft"),
    ("§7 annotated", f"{int(rb['annotation_progress']['annotated'].sum())} of {int(rb['annotation_progress']['sampled'].sum())}", "0 of 300"),
    ("§5 masking available", str(rb["masking_available"]), "False"),
])

number quoted in the text,recomputed,quoted,ok
§1 most frequent placeholder,"(18,765)","(18,765)",✓
§2 before: name / domain label / company / address,41% / 44% / 27% / 15%,41% / 44% / 27% / 15%,✓
§2 after: name + address + company,0,0,✓
§2 after: domain label,"3 of 12,536","3 of 12,536",✓
§3 probe raw / anonymised,12.2% / 3.1%,12.2% / 3.1%,✓
§3 fair chance level,2.1%,2.1%,✓
§3 validation trajectories,"1,723","1,723",✓
§3 times chance raw / anonymised,5.9 / 1.5,5.9 / 1.5,✓
§3 accuracy lost,75%,75%,✓
§3 neighbour of the same type,79%,79%,✓


All 20 numbers quoted in the text match the data.
